In [ ]:
!uv pip install dscribe ase

In [ ]:
import warnings

# Stops some annoying warnings from coming up
warnings.filterwarnings("ignore", category=SyntaxWarning)

## The Coulomb Matrix Formula

For a molecule with $N$ atoms, the Coulomb matrix $\mathbf{C}$ is $N \times N$:

$$
C_{ij} = \begin{cases}
  0.5\, Z_i^{2.4} & i = j \\[6pt]
  \dfrac{Z_i\, Z_j}{|\mathbf{R}_i - \mathbf{R}_j|} & i \neq j
\end{cases}
$$

- **Diagonal** — encodes _what atom_ this is (based on $Z$ = atomic number)
- **Off-diagonal** — encodes _how close_ two atoms are, weighted by their charges (like nuclear repulsion)

To use as an ML input, we create a 1D vector representation of $\mathbf{C}$ that doesn't change if you reorder the atoms.


# Manual Demo

Since this is pretty easy method to code up ourselves, we will first manually create the Coulomb Matrix and display it for a representative molecule.


In [ ]:
import numpy as np
from ase.atoms import Atoms


def get_coulomb(mol: Atoms) -> np.ndarray:
    """Return the raw N×N Coulomb matrix."""
    Z = mol.get_atomic_numbers()
    pos = mol.get_positions()
    N = len(Z)
    C = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if i == j:
                C[i, j] = 0.5 * Z[i] ** 2.4
            else:
                C[i, j] = Z[i] * Z[j] / np.linalg.norm(pos[i] - pos[j])
    return C

Let's start by viewing the Coulomb matrix of C60. First, we create the `Atoms` object.


In [ ]:
from ase.build import molecule
from ase.visualize import view

mol = molecule("C60")
view(mol, viewer="x3d")

Then we call our function to generate the Coulomb matrix, which we plot.


In [ ]:
import matplotlib.pyplot as plt

C = get_coulomb(mol)
plt.imshow(C)

Now it's your turn. Take a minute to construct Coulomb matrices using the example molecules available with `ase.build.molecule` (the list of supported molecules can be found [here](https://ase-lib.org/ase/build/build.html#ase.build.molecule)). Can you rationalize any of the features that appear in the Coulomb matrix?


In [ ]:
# your code here

# DScribe


There is a code `dscribe` that contains several structural feature generation routines. We can call it directly. Let's consider the following molecules: H2, H2O, NH3, and C60. We will make the Coulomb matrix for each of these using the methods within `dscribe`. For additional details of how the `CoulombMatrix` works in `dscribe`, refer to [the documentation](https://singroup.github.io/dscribe/latest/tutorials/descriptors/coulomb_matrix.html).


In [ ]:
molecules = [molecule("H2"), molecule("H2O"), molecule("NH3"), molecule("C60")]

In [ ]:
from dscribe.descriptors import CoulombMatrix

max_N = max([len(mol) for mol in molecules])
cm = CoulombMatrix(
    n_atoms_max=max_N,  # we need to pad the Coulomb matrices with 0s based on the largest structure
    permutation="eigenspectrum",  # we need to flatten the matrix to a 1D vector in some way
)

for mol in molecules:
    C = cm.create(mol)  # this creates the Coulomb matrix, C
    print(mol.get_chemical_formula())
    print(C)

# Reproducing a Paper

In this exercise, we will reproduce the resuls of the original [Coulomb matrix paper](https://link.aps.org/doi/10.1103/PhysRevLett.108.058301). Specifically, we will aim to predict the atomization energy of small organic molecules in the QM7 dataset.


First, we need to load in the dataset. These details are not important. The only thing that matters is that we will define `molecules` as our list of `Atoms` objects and `y` as our desired property (atomization energy).


In [ ]:
import os
import urllib.request
from ase.atoms import Atoms
from scipy.io import loadmat

url = "http://quantum-machine.org/data/qm7.mat"
filename = "qm7.mat"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)


data = loadmat(filename)
R = data["R"]  # atomic coordinates
Z = data["Z"]  # atomic numbers
y = data["T"].reshape(-1)  # atomization energies

molecules = []
for i in range(len(y)):
    mask = Z[i] > 0
    positions = R[i][mask]
    numbers = Z[i][mask]

    molecules.append(Atoms(numbers=numbers, positions=positions))

In [ ]:
print(len(molecules))
print(len(y))

Let's take a look at the distribution of properties we want to predict to see if there are any anomalies to keep in mind.


In [ ]:
import matplotlib.pyplot as plt

plt.hist(y, bins=50, edgecolor="k")
plt.xlabel("Atomization energy (kcal/mol)")
plt.ylabel("Count")

Now we will construct the Coulomb matrices using dscribe. We will flatten each matrix by taking the sorted eigenvalues. There are other options available via the `permutation` keyword argument as noted in [the documentation](https://singroup.github.io/dscribe/latest/tutorials/descriptors/coulomb_matrix.html#options-for-permutation).


In [ ]:
max_atoms = max([len(mol) for mol in molecules])

cm = CoulombMatrix(
    n_atoms_max=max_atoms,  # we need to pad the Coulomb matrices with 0s based on the largest structure
    permutation="eigenspectrum",  # we need to flatten the matrix to a 1D vector in some way
)
X = cm.create(
    molecules,  # this is our list of `Atoms` objects to featurize
    n_jobs=-1,  # this tells `dscribe` to run in parallel across our `Atoms` objects
)

In [ ]:
X.shape

Now let's reproduce Figure 1a in the paper, in which we calculate the Euclidean distance metric for each pair of molecules.


In [ ]:
from sklearn.metrics import pairwise_distances

# NxN matrix describing distance in feature space
D = pairwise_distances(X, metric="euclidean", n_jobs=-1)

In [ ]:
D.shape

In [ ]:
plt.hist(D.flatten(), bins=100, edgecolor="k")
plt.xlabel("d")
plt.ylabel("Frequency")

Now let's use the flattened Coulomb matrices to train a KRR model. For the sake of simplicity, we will do a bad thing and only have a training and testing set, with no validation set. We will assume some KRR hyperparameters are good. In practice, these hyperparameters would need to be tuned by minimizing the error in a validation set.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

In [ ]:
from sklearn.kernel_ridge import KernelRidge

model = KernelRidge(kernel="rbf", alpha=1e-5, gamma=1e-3)

model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

print(f"Test MAE: {mae} kcal/mol")

Now we reproduce the parity plot in Figure 2b. Look sgood!


In [ ]:
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.ylabel("ML-Predicted Atomization Energy (kcal/mol)")
plt.xlabel("Ground Truth Atomization Energy (kcal/mol)")

# Exercise

Try to reproduce the learning curve shown in Figure 2a. Note that the x-axis is actually N, not log2(N). The spacing is just on a log spacing.


In [ ]:
# code goes here